Harmonization 

In [1]:
# uninstalled dependencies
%pip install datasketch hnswlib sentence-transformers
import math
import unicodedata  
from collections import Counter, defaultdict
from itertools import combinations
import re
import numpy as np
import pandas as pd
from datasketch import MinHash, MinHashLSH
import hnswlib
from sentence_transformers import SentenceTransformer

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.3/780.3 kB 47.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.0/857.0 kB 85.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 121.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 90.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 101.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 808.8/808.8 kB 51.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 539.2/539.2 MB 58.1 MB/s  0:00:05m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.2/519.2 MB 59.8 MB/s  0:00:05m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 102.5 MB/s  0:00:010:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.0/216.0 MB 105.3 MB/s  0:00:020:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# Config

GOLD_FILE = 'PATSTAT_500_label_datav4.csv'
NAME_COL = 'comp'
GROUP_COL = 'Group'

EMBEDDING_MODEL = "Vsevolod/company-names-similarity-sentence-transformer" # embedding model purpose-built for company names
# MinHash parameters
MINHASH_THRESHOLD     = 0.2
MINHASH_PERM          = 128
SHINGLE_K             = 3

# HNSW parameters
HNSW_M                = 16
HNSW_K                = 15
HNSW_EF_CONSTRUCTION  = 100
HNSW_EF_SEARCH        = 100

# First-Token parameters
MAX_FIRST_TOKEN_BUCKET = 500

# sample size for the quick recall probe during tuning; None = full
TUNING_SAMPLE = None

Parte generale per le firms -> dopo vedo come adattarla quando ho anche uni e individui

### Parte di clening del nome 
va bene per tutto uni e firms 

In [3]:
FORMS = ["INC", "INCORPORATED", "LLC", "LLP", "LP", "LTD", "LIMITED", "CORP", "CORPORATION", "CO", "COMPANY", "PLC", "GMBH", "MBH",
         "AG", "KG", "SE", "SA", "SAS", "SARL", "SRL", "SPA", "SL", "BV", "NV", "OY", "OYJ", "AB", "AS", "ASA", "APS", "PTY", "PTE",
         "KK", "KABUSHIKI KAISHA", "SP Z OO", "OOO", "BHD", "SDN", "PVT", "LTDA"]
FORMS_RE = "|".join(sorted(FORMS, key=len, reverse=True))     # longest first: 'CORPORATION' before 'CO'
FORMS_CJK = "股份有限公司|有限责任公司|有限責任公司|有限公司|株式会社|有限会社|合同会社|주식회사|公司"
WORDS = {"UNIVERSITY": "UNIV", "UNIVERSITAET": "UNIV", "UNIVERSITAT": "UNIV", "UNIVERSITE": "UNIV", "UNIVERSIDAD": "UNIV",
         "UNIVERSITA": "UNIV", "INSTITUTE": "INST", "INSTITUT": "INST", "INSTITUTO": "INST", "TECHNOLOGY": "TECH",
         "TECHNOLOGIES": "TECH", "TECHNOLOGICAL": "TECH", "LABORATORY": "LAB", "LABORATORIES": "LAB", "LABS": "LAB",
         "CENTRE": "CENTER", "SCIENCES": "SCIENCE", "大學": "大学"}
FILLER = {"OF", "THE", "FOR", "AND", "DE", "DI", "DU", "DES", "ET", "UND"}

ISO2 = {
    "US","GB","UK","DE","FR","JP","CN","KR","CA","AU","IT","NL","SE","CH","IL",
    "IN","BR","RU","ES","BE","AT","DK","FI","NO","TW","IE","SG","HK","NZ","ZA",
    "MX","AR","PL","CZ","HU","PT","GR","TR","MY","TH","PH","ID","VN","SA","AE",
    "EG","LU","IS","CY","EE","LV","LT","SI","SK","BG","RO","HR","RS","UA"
}

COUNTRY_NAMES = {
    "USA","UNITED STATES","UNITED STATES OF AMERICA","UK","UNITED KINGDOM",
    "GREAT BRITAIN","ENGLAND","SCOTLAND","WALES","GERMANY","DEUTSCHLAND","FRANCE"
    ,"JAPAN","NIPPON","NIHON","CHINA","PEOPLES REPUBLIC OF CHINA","PRC","KOREA",
    "SOUTH KOREA","REPUBLIC OF KOREA","TAIWAN","CANADA","AUSTRALIA","ITALY",
    "ITALIA","NETHERLANDS","HOLLAND","SWEDEN","SWITZERLAND","ISRAEL","INDIA",
    "BRAZIL","BRASIL","RUSSIA","RUSSIAN FEDERATION","SPAIN","BELGIUM","AUSTRIA",
    "DENMARK","FINLAND","NORWAY","IRELAND","SINGAPORE","HONG KONG","NEW ZEALAND",
    "MEXICO","ARGENTINA","POLAND","CZECH REPUBLIC","HUNGARY","PORTUGAL","GREECE",
    "TURKEY","MALAYSIA","THAILAND","PHILIPPINES","INDONESIA","VIETNAM","SAUDI ARABIA",
    "UNITED ARAB EMIRATES","LUXEMBOURG","ICELAND"
}

def _is_country(tok):
    return tok in ISO2 or tok in COUNTRY_NAMES

def strip_country(s):
    if not isinstance(s, str) or not s.strip():
        return ""
    s = re.sub(r"[^A-Z0-9 ,]", " ", s.upper())
    s = re.sub(r"\s+", " ", s).strip()
    if "," in s:
        head, tail = s.rsplit(",", 1)
        tt = tail.strip().split()
        if 1 <= len(tt) <= 4 and all(_is_country(t) for t in tt):
            s = head.strip()
    toks = s.split()
    if len(toks) >= 2:
        for n in (4, 3, 2, 1):
            if len(toks) >= n + 1:
                cand = " ".join(toks[-n:])
                if cand in COUNTRY_NAMES or (n == 1 and toks[-1] in ISO2):
                    toks = toks[:-n]
                    s = " ".join(toks)
                    break
    return re.sub(r"\s+", " ", s.replace(",", " ")).strip()


def clean(name):
    s = unicodedata.normalize("NFKD", unicodedata.normalize("NFKC", str(name)).upper())
    s = unicodedata.normalize("NFC", "".join(c for i, c in enumerate(s) if not (unicodedata.combining(c) and s[i - 1] < "\u0250")))
    s = re.sub(r"[▲▼?.'’]", "", s)
    latin = re.search(r"[^\x00-\x7f]([A-Z0-9][A-Z0-9 ,&\-/]+)$", s)     # CJK name + Latin version -> Latin part
    if latin and len(re.sub(r"[^A-Z]", "", latin.group(1))) >= 3:
        s = latin.group(1)
    s = re.sub(r"\bC/O\b.*$", "", s).replace("A/S", "AS").replace("&", " AND ")
    s = re.sub(rf"^({FORMS_CJK})|({FORMS_CJK})$", "", re.sub(r"[^\w ]|_", " ", s).strip())
    s = " ".join(WORDS.get(w, w) for w in s.split() if w not in FILLER) or s
    s = re.sub(rf"^(({FORMS_RE}) )+|( ({FORMS_RE}))+$", "", s).strip() or s
    return strip_country(s) or s  # [AGGIUNTO] chiamata a strip_country() che prima era definita ma non usata

# [AGGIUNTO] Cella/blocco mancante per caricare il gold standard e creare la lista 'names'
df = pd.read_csv(GOLD_FILE)
df['clean'] = df[NAME_COL].apply(clean)
df = df[df['clean'].str.len() > 0].copy()  # rimuove eventuali nomi vuoti
names = sorted(df['clean'].unique())
print(f"names: {len(names):,}  groups: {df[GROUP_COL].nunique():,}")

names: 2,911  groups: 481


Proposta delle coppie -> non posso veirficare 2 a 2 tutti i nomi 
Faccio delle plausibili coppie di nomi simili 
3 tecniche:
1) errori di battitura--> 3 lettere consecutive uguali (secondo me possiamo alzare?), abbimao un paper di riferimento che ci dice 3? --> è troppo lento 3  a 3 faccio delle cose strane non capite per andare veloce. C'è riferimento alla letteratura
2) vicinanza semantica anche qui fatto dal paper
3) prima parola uguale 
4) seavessimo han possiamo anche proporre chi ha quello uguale (nel goal standard non c'è, nel mio settore sì) 

Controlla cosa c'era nel paper -> se non c'è nel paper si leva !!!
Fra le coppie suggerite faccio un confronto di similarità a 3 pesi, spiegato dopo

In [4]:
# STAGE 1 (MinHash + SBERT and HNSW: from LSBlock): functions

def shingles(name, k=SHINGLE_K):
    s = name.lower()
    return {s[i:i+k] for i in range(len(s) - k + 1)} if len(s) >= k else {s}

def minhash_candidates(names):
    lsh = MinHashLSH(threshold=MINHASH_THRESHOLD, num_perm=MINHASH_PERM)
    sketches = {}
    for n in names:
        m = MinHash(num_perm=MINHASH_PERM)
        for sh in shingles(n):
            m.update(sh.encode("utf8"))
        sketches[n] = m
        lsh.insert(n, m)
    pairs = set()
    for n, m in sketches.items():
        for other in lsh.query(m):
            if other != n:
                pairs.add((n, other) if n < other else (other, n))
    return pairs

def semantic_candidates(names, model, k=HNSW_K):
    emb = model.encode(names, normalize_embeddings=True, show_progress_bar=True)
    index = hnswlib.Index(space="cosine", dim=emb.shape[1])
    index.init_index(max_elements=len(names),
                     ef_construction=HNSW_EF_CONSTRUCTION, M=HNSW_M)
    index.add_items(emb, np.arange(len(names)))
    index.set_ef(HNSW_EF_SEARCH)
    labels, _ = index.knn_query(emb, k=min(k + 1, len(names)))
    pairs = set()
    for i, neigh in enumerate(labels):
        for j in neigh:
            if j == i:
                continue
            a, b = names[i], names[int(j)]
            pairs.add((a, b) if a < b else (b, a))
    return pairs, emb

def blocking_recall(df, name_col, group_col, candidate_set):
    groups = df.groupby(group_col)[name_col].apply(lambda s: sorted(set(s.dropna())))
    total, covered, missed = 0, 0, []
    for members in groups:
        for a, b in combinations(members, 2):
            total += 1
            pair = (a, b) if a < b else (b, a)
            if pair in candidate_set:
                covered += 1
            else:
                missed.append(pair)
    return covered / total if total else 0.0, total, missed

def first_token_candidates(names, max_bucket=MAX_FIRST_TOKEN_BUCKET):
    buckets = defaultdict(list)
    for n in names:
        toks = n.split()
        if toks:
            buckets[toks[0]].append(n)
    pairs = set()
    for members in buckets.values():
        if 2 <= len(members) <= max_bucket:
            for a, b in combinations(members, 2):
                pairs.add((a, b) if a < b else (b, a))
    return pairs
    
# STAGE 1: MinHash

mh_pairs = minhash_candidates(names)
print(f"minhash pairs: {len(mh_pairs):,}")


# STAGE 1: SBERT + HNSW

model = SentenceTransformer(EMBEDDING_MODEL)
#model = SentenceTransformer()
hn_pairs, emb = semantic_candidates(names, model)
print(f"hnsw pairs: {len(hn_pairs):,}")

# [AGGIUNTO] Mancava la chiamata a first_token_candidates e l'unione 'union' usata nello Stage 2
ft_pairs = first_token_candidates(names)
union = mh_pairs | hn_pairs | ft_pairs
recall, total, missed = blocking_recall(df, "clean", GROUP_COL, union)
print(f"first-token pairs: {len(ft_pairs):,} | union pairs: {len(union):,} | blocking recall: {recall:.2%}")

minhash pairs: 134,966


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/2.35k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/720 [00:00<?, ?B/s]

... (8 outputs left out)

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/91 [00:00<?, ?it/s]

hnsw pairs: 33,447
first-token pairs: 72,561 | union pairs: 174,195 | blocking recall: 97.76%


### Parte 2 creo il grafo pesato 
Collego ogni coppia trovato con un arco pesato -> peso dato in base a:
1) similarità di sbert --> semntica pura
2) jaccard --> simile le parole ma non conta ordine (similarità token)
3) non volgio essere overconfident -> metto anche un confronto lettera per lettera con peso bassino
   

In [5]:
import networkx as nx
import networkx.algorithms.community as nx_comm
from difflib import SequenceMatcher

# STAGE 2: config (if ok merge with top config cell)

SCORE_WEIGHTS = {"cosine": 0.55, "jaccard": 0.30, "edit": 0.15}
SWEEP_PERCENTILES = (50, 60, 70, 75, 80, 85, 88, 90, 92, 95, 97, 98)
LOUVAIN_RESOLUTION = 1.0

In [6]:
# STAGE 2: Feature builder and scoring

def pair_features(pairs, emb_dict):
    rows = []
    for a, b in pairs:
        ea, eb = emb_dict[a], emb_dict[b]
        ta, tb = set(a.split()), set(b.split())
        rows.append([
            float(np.dot(ea, eb)),
            len(ta & tb) / len(ta | tb) if (ta | tb) else 0.0,
            SequenceMatcher(None, a, b).ratio(),
            ])
    df = pd.DataFrame(rows, columns=["cosine", "jaccard", "edit"])
    df["name_a"] = [a for a, b in pairs]
    df["name_b"] = [b for a, b in pairs]
    df["score"] = sum(df[k] * w for k, w in SCORE_WEIGHTS.items())
    return df

In [7]:
# STAGE 2: scored candidate set

emb_dict = dict(zip(names, emb))
union_list = list(union)
scored = pair_features(union_list, emb_dict)
print(f"scored {len(scored):,} candidate pairs")
print(f"score distribution: min={scored['score'].min():.3f}  "
      f"median={scored['score'].median():.3f}  "
      f"p90={np.percentile(scored['score'], 90):.3f}  "
      f"max={scored['score'].max():.3f}")


# small correctness check
missing = [n for n in names if n not in emb_dict]
assert not missing, f"{len(missing)} names missing from emb_dict"
assert len(scored) == len(union_list), "scored must match union size"

scored 174,195 candidate pairs
score distribution: min=-0.153  median=0.219  p90=0.685  max=0.995


### Taglio del grafo 
Dopo che ho il grafo connesso toglio il peso degli archi e tengo solo le sotto sezioni a cui do il nome della company con più families 

In [20]:
import numpy as np
import networkx as nx
import networkx.algorithms.community as nx_comm

def find_threshold_otsu(scores, bins=100):
    """Trova la soglia ottimale che separa match e non-match minimizzando la varianza intra-classe."""
    # Consideriamo solo la fascia plausibile (> 0.35) per evitare il rumore di fondo del blocking
    s = scores[scores >= 0.35]
    hist, bin_edges = np.histogram(s, bins=bins)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2.0

    weight1 = np.cumsum(hist)
    weight2 = np.cumsum(hist[::-1])[::-1]

    mean1 = np.cumsum(hist * bin_centers) / np.maximum(weight1, 1)
    mean2 = (np.cumsum((hist * bin_centers)[::-1]) / np.maximum(weight2[::-1], 1))[::-1]

    # Varianza inter-classe tra le due distribuzioni
    variance_between = weight1[:-1] * weight2[1:] * (mean1[:-1] - mean2[1:]) ** 2
    idx = np.argmax(variance_between)
    return float(bin_centers[idx])



# Esecuzione unsupervised
th_otsu = find_threshold_otsu(scored["score"].values)

print(f"Soglia Otsu (unsupervised): {th_otsu:.3f}")


Soglia Otsu (unsupervised): 0.572


In [19]:
import networkx as nx
import networkx.algorithms.community as nx_comm
from math import comb

# 1. Soglia fissa
TH = 0.572

# 2. Costruisci il grafo e trova le comunità (Louvain non chiede il numero di gruppi)
G = nx.Graph()
G.add_nodes_from(names)
edges = scored[scored["score"] >= TH]
G.add_weighted_edges_from(zip(edges["name_a"], edges["name_b"], edges["score"]))

communities = nx_comm.louvain_communities(G, weight="weight", seed=42)

# 3. Scegli automaticamente un nome rappresentativo per ogni cluster (il più frequente, poi il più corto)
counts = df["clean"].value_counts()
harmonized_map = {}
for members in communities:
    best_name = min(members, key=lambda n: (-counts.get(n, 0), len(n)))
    for m in members:
        harmonized_map[m] = best_name

# Aggiungi la nuova colonna armonizzata al DataFrame del gold standard
df["harmonized"] = df["clean"].map(harmonized_map)

# 4. Metriche esatte della prof (Pairwise Precision, Recall, F1)
def pairs_inside(group_sizes):
    return sum(comb(int(n), 2) for n in group_sizes)

def precision_recall_f1(gold_groups, predicted_groups):
    res_df = pd.DataFrame({"gold": list(gold_groups), "pred": list(predicted_groups)})
    tp = pairs_inside(res_df.groupby(["gold", "pred"]).size())
    tp_fp = pairs_inside(res_df.groupby("pred").size())
    tp_fn = pairs_inside(res_df.groupby("gold").size())
    precision = tp / tp_fp if tp_fp else 1.0
    recall = tp / tp_fn if tp_fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return {"groups found": res_df["pred"].nunique(), "precision": precision, "recall": recall, "F1": f1}

# Esecuzione della verifica e stampa dei risultati
print(f"Gruppi veri nel gold standard: {df[GROUP_COL].nunique()}")
print(pd.Series(precision_recall_f1(df[GROUP_COL], df["harmonized"])).round(3))

# Anteprima delle colonne finali
display(df[[NAME_COL, GROUP_COL, "harmonized"]].head(15))

Gruppi veri nel gold standard: 481
groups found    569.000
precision         0.920
recall            0.854
F1                0.886
dtype: float64


,comp,Group,harmonized
0,NOKIA CORPORATION,1,NOKIA
1,NOKIA (DEUTSCHLAND),1,NOKIA
2,NOKIA GRAETZ,1,NOKIA
3,NOKIA SIEMENS NETWORKS SCHWEIZ,2,NOKIA
4,NOKIA,1,NOKIA
5,NOKIA MOBILE PHONES (U.K.) LIMITED,1,NOKIA
6,NOKIA SIEMENS NETWORKS S.P.A.,2,NOKIA
7,NOKIA SIEMENS NETWORKS,2,NOKIA
8,NOKIA COMMUNICATION,1,NOKIA
9,NOKIA HIGH SPEED ACCESS PRODUCTS,1,NOKIA
